# Phase 13b: Model Evaluation — Confusion Matrix, ROC, Feature Importance

Confusion matrix, ROC curve, and top-10 feature importances for the tuned XGBoost model.

## Setup

In [1]:
import numpy as np, pandas as pd, joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, roc_curve, auc

best_model = joblib.load('../models/best_model_xgboost_tuned.joblib')
X_train, X_test, y_train, y_test = joblib.load('../data/train_test_split.joblib')

y_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

## Confusion matrix

In [2]:
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Not High-Value', 'High-Value'],
                                          cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Confusion Matrix — Tuned XGBoost')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png', dpi=150)
plt.close()

## ROC curve

In [3]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc:.3f})', linewidth=2, color='#2563eb')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Tuned XGBoost')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/roc_curve.png', dpi=150)
plt.close()

## Top 10 feature importances

In [4]:
preprocessor = best_model.named_steps['preprocessor']
clf = best_model.named_steps['classifier']

num_features = preprocessor.transformers_[0][2]
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_features = list(cat_encoder.get_feature_names_out(preprocessor.transformers_[1][2]))
all_features = list(num_features) + cat_features

importances = clf.feature_importances_
imp_df = pd.DataFrame({'feature': all_features, 'importance': importances}) \
    .sort_values('importance', ascending=False).head(10)

print("Top 10 most important features:")
print(imp_df.to_string(index=False))
imp_df.to_csv('../outputs/top10_feature_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color='#2563eb')
ax.set_xlabel('Importance')
ax.set_title('Top 10 Feature Importances — Tuned XGBoost')
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150)
plt.close()

print("\nSaved: confusion_matrix.png, roc_curve.png, feature_importance.png, top10_feature_importance.csv")

Top 10 most important features:
                feature  importance
                mass_kg    0.127342
product_code_grp_040043    0.086612
 product_code_grp_12253    0.042118
  product_code_grp_7107    0.039603
 product_code_grp_10574    0.036996
 product_code_grp_11522    0.033311
 product_code_grp_12059    0.033172
 product_code_grp_10314    0.032873
 product_code_grp_12028    0.032438
 product_code_grp_OTHER    0.029055

Saved: confusion_matrix.png, roc_curve.png, feature_importance.png, top10_feature_importance.csv
